In [ ]:
Acne Detection & Skin Analyzer

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
IMAGE_PATH = "Test.JPG"

img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise ValueError("Image not found")

img_bgr = cv2.resize(img_bgr, (420,420))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.title("Input")
plt.axis("off")

In [ ]:
# preprocess 
lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
l,a,b = cv2.split(lab)

clahe = cv2.createCLAHE(2.0,(8,8))
l = clahe.apply(l)

img_bgr = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)

# sharpen
kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
img_bgr = cv2.filter2D(img_bgr,-1,kernel)

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.title("Enhanced")

In [ ]:
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

h,s,v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
a = lab[:,:,1].astype(float)

In [ ]:
# local redness
blur = cv2.GaussianBlur(a,(0,0),5)
redness = a - blur

# texture
lap = np.abs(cv2.Laplacian(gray,cv2.CV_32F))

# blob detection
g1 = cv2.GaussianBlur(gray.astype(float),(0,0),1)
g2 = cv2.GaussianBlur(gray.astype(float),(0,0),3)
dog = np.abs(g1-g2)

plt.imshow(redness,cmap='hot'); plt.title("Redness")

In [ ]:
# skin filter
skin = ((h<25)&(s>20)&(v>50)).astype(np.uint8)*255

# focus mask
mask_focus = np.zeros_like(gray)
hgt,w = gray.shape

cv2.ellipse(mask_focus,(w//2,int(hgt*0.6)),(int(w*0.25),int(hgt*0.3)),0,0,360,255,-1)

mask_focus[:int(hgt*0.2),:] = 0
mask_focus[int(hgt*0.7):,:] = 0

plt.imshow(mask_focus,cmap='gray'); plt.title("Focus")

In [ ]:
candidate = (
    (redness>3.0) &
    (dog>4.0) &
    (lap>7.0) &
    (skin>0) &
    (mask_focus>0)
).astype(np.uint8)*255

plt.imshow(candidate,cmap='gray'); plt.title("Candidate")

In [ ]:
# clean + filter blobs
num, labels, stats, _ = cv2.connectedComponentsWithStats(candidate)

mask = np.zeros_like(candidate)

for i in range(1,num):
    area = stats[i,cv2.CC_STAT_AREA]
    w = stats[i,cv2.CC_STAT_WIDTH]
    h = stats[i,cv2.CC_STAT_HEIGHT]

    if h == 0: continue

    ar = w/h
    if 5 < area < 60 and 0.5 < ar < 2:
        mask[labels==i] = 255

plt.imshow(mask,cmap='gray'); plt.title("Filtered")

In [ ]:
overlay = img_rgb.copy()
overlay[mask>0] = [255,0,0]

plt.imshow(overlay)
plt.title("Result")
plt.axis("off")

In [ ]:
spots = cv2.connectedComponents(mask)[0]-1
area = np.sum(mask>0)
total = np.sum(mask_focus>0)

# cap values to realistic ranges
norm_percent = min(percent / 15.0, 1.0)   # 15% = very severe
norm_spots   = min(spots / 40.0, 1.0)     # 40 spots = severe

# weighted score (0–100)
score = (
    0.6 * norm_percent +
    0.4 * norm_spots
) * 100

# classification
if score < 70:
    label = "Mild"
elif score < 130:
    label = "Moderate"
else:
    label = "Severe"

print("Spots:", spots)
print("Area%:", percent)
print("Score:", round(score,2))
print("Severity:", label)